# Interactive GenBank ORF classification with XGBoost Random Forest

This notebook reproduces the core `genome_entropy ml` workflow interactively. It reads one pipeline JSON file containing multiple top-level sequence records, removes records with no usable ORFs, keeps every sequence's ORFs together during the train/test split, trains an XGBoost random forest, and evaluates predictions on held-out records.

The record-level split is important: an ORF from a sequence used for testing must not leak into training. The input therefore needs at least two top-level records containing usable ORFs.

## 1. Configuration

Set `JSON_PATH` to a `.json` or `.json.gz` file produced by `genome_entropy run`. Adjust the model and split settings as needed. XGBoost's `XGBRFClassifier` trains a random forest rather than the gradient-boosted model used by the default CLI classifier.

In [ ]:
from pathlib import Path

JSON_PATH = Path("/path/to/genome_entropy_results.json")
TEST_SPLIT = 0.20
RANDOM_SEED = 42

# XGBoost random-forest settings
N_ESTIMATORS = 500
MAX_DEPTH = 8
SUBSAMPLE = 0.80
COLSAMPLE_BYNODE = 0.80
N_JOBS = -1
DEVICE = "cpu"  # Change to "cuda" when GPU-enabled XGBoost is available.

# Optional outputs
PLOT_FEATURE_IMPORTANCE = True
MODEL_OUTPUT = None  # For example: Path("xgboost_random_forest.json")

## 2. Imports

This notebook reuses the package's JSON parsing and feature extraction code so its input handling stays consistent with `genome_entropy ml`. From the repository root, install its complete runtime environment with `pip install -e '.[ml]' jupyterlab pandas matplotlib` if needed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
)
from xgboost import XGBRFClassifier

from genome_entropy.ml import (
    extract_features,
    filter_json_records_with_features,
    load_json_file,
    split_json_records,
)

pd.set_option("display.max_columns", None)

## 3. Read and validate the multi-record JSON file

`load_json_file` accepts both plain and gzip-compressed JSON. Each top-level record represents one input sequence. Empty pipeline results and records whose ORFs cannot provide the required ML features are removed before splitting.

In [ ]:
if not JSON_PATH.is_file():
    raise FileNotFoundError(f"Set JSON_PATH to an existing file: {JSON_PATH}")

all_records = load_json_file(JSON_PATH)
usable_records = filter_json_records_with_features(all_records)

print(f"Top-level records read: {len(all_records):,}")
print(f"Records with usable ORFs: {len(usable_records):,}")
print(f"Featureless records removed: {len(all_records) - len(usable_records):,}")

if len(usable_records) < 2:
    raise ValueError(
        "At least two top-level records containing usable ORFs are required "
        "for record-level training and testing."
    )

## 4. Split records and extract ORF features

The random split is reproducible. Splitting occurs before feature extraction, at the top-level record boundary, so all ORFs from one sequence remain entirely in training or entirely in testing.

In [ ]:
train_records, test_records = split_json_records(
    usable_records,
    test_split=TEST_SPLIT,
    random_seed=RANDOM_SEED,
)

X_train, y_train, feature_names, train_metadata = extract_features(
    train_records, return_metadata=True
)
X_test, y_test, test_feature_names, test_metadata = extract_features(
    test_records, return_metadata=True
)

if feature_names != test_feature_names:
    raise ValueError("Training and test feature columns do not match")

print(f"Training records: {len(train_records):,}")
print(f"Testing records: {len(test_records):,}")
print(f"Training ORFs: {len(X_train):,}")
print(f"Testing ORFs: {len(X_test):,}")
print(f"Features per ORF: {len(feature_names):,}")

if np.unique(y_train).size < 2:
    raise ValueError(
        "The training records contain only one in_genbank class. Adjust "
        "TEST_SPLIT or RANDOM_SEED, or provide records containing both classes."
    )

## 5. Explore the feature matrices and class balance

The target is `in_genbank`: `1` means the ORF matched a GenBank CDS annotation and `0` means it did not.

In [ ]:
train_frame = pd.DataFrame(X_train, columns=feature_names)
train_frame["in_genbank"] = y_train
test_frame = pd.DataFrame(X_test, columns=feature_names)
test_frame["in_genbank"] = y_test

display(train_frame.head())
display(train_frame.describe().T)

class_balance = pd.DataFrame(
    {
        "training": pd.Series(y_train).value_counts().sort_index(),
        "testing": pd.Series(y_test).value_counts().sort_index(),
    }
).fillna(0).astype(int)
class_balance.index = class_balance.index.map({0: "not_in_genbank", 1: "in_genbank"})
display(class_balance)

## 6. Build and train the XGBoost random forest

`XGBRFClassifier` creates many randomized trees in parallel. `subsample` controls row sampling and `colsample_bynode` controls feature sampling at each node. Increase `n_estimators` for a larger forest; tune depth and sampling to control complexity.

In [ ]:
model = XGBRFClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    subsample=SUBSAMPLE,
    colsample_bynode=COLSAMPLE_BYNODE,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    device=DEVICE,
    random_state=RANDOM_SEED,
    n_jobs=N_JOBS,
)

model.fit(X_train, y_train)
model

## 7. Test the model on held-out records

Accuracy summarizes all predictions, while precision, recall, and F1 are useful when GenBank annotation classes are imbalanced. ROC AUC is calculated only when the held-out set contains both classes.

In [ ]:
test_predictions = model.predict(X_test).astype(int)
test_probabilities = model.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test,
    test_predictions,
    labels=[0, 1],
    target_names=["not in GenBank", "in GenBank"],
    zero_division=0,
))

if np.unique(y_test).size == 2:
    print(f"ROC AUC: {roc_auc_score(y_test, test_probabilities):.4f}")
else:
    print("ROC AUC is undefined because the held-out records contain one class.")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=["not in GenBank", "in GenBank"],
    cmap="Blues",
)
plt.title("Held-out ORF predictions")
plt.tight_layout()
plt.show()

## 8. Inspect every held-out ORF prediction

Including both `input_id` and `orf_id` keeps predictions identifiable when ORF names repeat between sequence records. Sort or filter this table interactively to investigate errors and uncertain predictions.

In [ ]:
prediction_frame = pd.DataFrame(test_metadata)
prediction_frame["actual_label"] = y_test
prediction_frame["predicted_label"] = test_predictions
prediction_frame["probability_in_genbank"] = test_probabilities
prediction_frame["correct"] = y_test == test_predictions

display(prediction_frame.head(20))
display(prediction_frame.loc[~prediction_frame["correct"]].head(20))

## 9. Optional variable-importance plot

XGBoost's `feature_importances_` values summarize each variable's contribution across the forest. They show association with model decisions, not causation. Correlated variables can share or redistribute importance.

In [ ]:
importance_frame = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "importance": model.feature_importances_,
        }
    )
    .sort_values("importance", ascending=True)
    .reset_index(drop=True)
)

display(importance_frame.sort_values("importance", ascending=False))

if PLOT_FEATURE_IMPORTANCE:
    ax = importance_frame.plot.barh(
        x="feature",
        y="importance",
        figsize=(9, 6),
        legend=False,
        color="steelblue",
    )
    ax.set_title("XGBoost random-forest variable importance")
    ax.set_xlabel("Importance")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

## 10. Optionally save the trained model

XGBoost's JSON model format is portable across supported XGBoost versions and retains the tree ensemble. The feature order remains the `feature_names` list shown above.

In [ ]:
if MODEL_OUTPUT is not None:
    MODEL_OUTPUT = Path(MODEL_OUTPUT)
    MODEL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(MODEL_OUTPUT)
    print(f"Saved model to {MODEL_OUTPUT.resolve()}")
else:
    print("MODEL_OUTPUT is None; the model was not written to disk.")